# You Cannot Subtract a Mean You Cannot See

**Topic:** Kernel centering &nbsp;·&nbsp; **Difficulty:** Hard

PCA starts by subtracting the mean. Kernel PCA works in a feature space you never
build, so there is no mean vector to subtract, but the centering still has to
happen, and it can be done entirely in terms of the kernel matrix.

The identity is `K_c = K - 1_n K - K 1_n + 1_n K 1_n` where `1_n` is the matrix
of all `1/n`. Implement it, and verify against an explicit feature map where you
*can* see the mean.

## The data (created in the next cell)

- `X` `(200, 3)`: the raw points.
- A polynomial feature map `phi` of degree 2 is available, so the kernel
  `K(a, b) = (a . b + 1)^2` has an explicit finite-dimensional realization.

## Your task

With **NumPy only**:

1. Build `K` `(200, 200)` from `K(a, b) = (a . b + 1)^2`.
2. Centre it with the double-centering identity to get `K_centered`.
3. Build the explicit feature matrix `Phi` using `phi`, centre it by subtracting
   the column mean, and form `Phi_c @ Phi_c.T`.
4. Set `answers` to a dict with `"max_diff"`, the largest absolute difference
   between `K_centered` and the explicit version, plus `"row_mean_absmax"`
   (largest absolute row mean of `K_centered`, which must be ~0),
   `"trace_before"`, `"trace_after"` (the traces of `K` and of
   `K_centered`, in that order).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the two centering routes agree and that the centered kernel has zero row
means.

In [ ]:
import numpy as np

rng = np.random.default_rng(3009)
X = rng.normal(0, 1, (200, 3))

def phi(Z):
    x, y, z = Z[:, 0], Z[:, 1], Z[:, 2]
    s2 = np.sqrt(2.0)
    return np.column_stack([
        x ** 2, y ** 2, z ** 2,
        s2 * x * y, s2 * x * z, s2 * y * z,
        s2 * x, s2 * y, s2 * z,
        np.ones(len(Z)),
    ])

print(f"X {X.shape}, and the degree-2 map lands in {phi(X).shape[1]} dimensions")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("max_diff", [], 4.263256414560601e-14, -7.138311906992065e-15, 1e-06),
    ("row_mean_absmax", [], 7.105427357601002e-15, -1.1897186511653442e-15, 1e-06),
    ("trace_before", [], 4142.28861446102, -693.5765823940554, 1e-06),
    ("trace_after", [], 3378.3828631606325, -565.6697198909401, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "max_diff" in answers:
    _agree = 0.0 < answers["max_diff"] < 1e-8
    _zero = answers["row_mean_absmax"] < 1e-10
    print(f"  {'matches explicit map':<24} {'ok' if _agree else 'wrong'}  "
          f"(max diff {answers['max_diff']:.2e})")
    print(f"  {'row means are zero':<24} {'ok' if _zero else 'wrong'}")
    _ok = _ok and _agree and _zero

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The identity comes from expanding `(phi(x_i) - mu) . (phi(x_j) - mu)` where
`mu` is the mean feature vector: the cross terms become row and column means of
`K`, and the `mu . mu` term becomes the grand mean. Nothing about it requires
knowing `phi`, which is the whole point: the same three lines centre a Gaussian
kernel whose feature space is infinite-dimensional. Forgetting to centre is a
common kernel-PCA bug and shows up as a spurious leading component that just
encodes the offset.

```python
import numpy as np

n = len(X)
K = (X @ X.T + 1.0) ** 2

one_n = np.full((n, n), 1.0 / n)
K_centered = K - one_n @ K - K @ one_n + one_n @ K @ one_n

Phi = phi(X)
Phi_c = Phi - Phi.mean(axis=0, keepdims=True)
K_explicit = Phi_c @ Phi_c.T

answers = {
    "max_diff": float(np.abs(K_centered - K_explicit).max()),
    "row_mean_absmax": float(np.abs(K_centered.mean(axis=1)).max()),
    "trace_before": float(np.trace(K)),
    "trace_after": float(np.trace(K_centered)),
}
```